## AMR Parsing using Phi-3.5

Finetuning framework from Unsloth Repository: https://github.com/unslothai/unsloth <br>
Evaluation framework crafted individually using SMATCH library: https://github.com/snowblink14/smatch <br>
Open source models: https://huggingface.co/unsloth

<b>Other Misc Libraries used <b> <br>
Transformers: https://github.com/huggingface/transformers <br>
Tensorboard: https://github.com/tensorflow/tensorboard <br>
Accelerate: https://github.com/huggingface/accelerate <br>
TQDM: https://github.com/tqdm/tqdm <br>
Pytorch: https://github.com/pytorch/pytorch <br>
Plotly: https://github.com/plotly <br>
Scipy: https://github.com/scipy/scipy <br>
Pandas: https://github.com/pandas-dev/pandas <br>
Datasets: https://github.com/huggingface/datasets <br>
Wandb: https://github.com/wandb/wandb <br>

### Download Model

In [1]:
%%capture
!python -m pip install --upgrade pip
!pip install unsloth "xformers==0.0.28.post2"
!pip uninstall unsloth -y && pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade --no-cache-dir --no-deps unsloth transformers git+https://github.com/huggingface/trl.git
!pip install transformers -U
!pip install tensorboardX
!pip install accelerate
!pip install tqdm
!pip install smatch
!wget https://raw.githubusercontent.com/snowblink14/smatch/master/smatch.py -O smatch.py
!pip install plotly scipy pandas
!pip install notebook ipywidgets
!pip install datasets
!pip install wandb

In [2]:
import wandb
wandb.login(key="6519d7308090d92747df2da031ddec59ba0d6d41")

wandb_config = {
    "learning_rate": 2e-4,
    "batch_size": 1,
    "grad_accumulation": 128,
    "model": "Phi 3.5",
    "dataset": "AMR",
    "epochs": 5
}

wandb.init(
    project="amr-parsing-best",
    name="phi-3.5-run",
    config=wandb_config
)

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from unsloth import FastLanguageModel, is_bfloat16_supported, unsloth_train
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset, Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, DataCollatorForLanguageModeling
from accelerate import Accelerator
import torch
import re
import subprocess
import time
import random
import numpy as np
import pandas as pd
from typing import List, Dict, Tuple, Any
from tqdm import tqdm
import plotly.io as pio
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

pio.renderers.default = "notebook"

In [ ]:
max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",     
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", 
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            
] 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    cache_dir = os.environ['TRANSFORMERS_CACHE']
)

### Data Prep

In [ ]:
dataset = load_dataset("hoshuhan/amr-3-parsed")

In [ ]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-3",
    mapping={
        "role": "role",
        "content": "content",
        "user": "user",
        "assistant": "assistant"
    }
)

In [10]:
def formatting_prompts_func(examples):
    formatted_conversations = []
    for conversation in examples["conversations"]:
        try:
            user_input = conversation[0]["content"].replace(
                "Generate an Abstract Meaning Representation (AMR) graph for the following sentence: ", 
                ""
            )
            
            formatted_chat = [
                {"role": "system", "content": "You are an AMR parser. Convert English sentences into Abstract Meaning Representation (AMR) graphs. Use proper AMR notation and formatting."},
                {"role": "user", "content": user_input},
                {"role": "assistant", "content": conversation[1]["content"]}
            ]
            
            formatted_text = tokenizer.apply_chat_template(
                formatted_chat,
                tokenize=True,
                add_generation_prompt=True,  
                return_tensors=None
            )
            formatted_conversations.append(formatted_text)
        except Exception as e:
            print(f"Error formatting conversation: {e}")
            continue
    
    return {"input_ids": formatted_conversations}

formatted_dataset = dataset.map(
    formatting_prompts_func,
    batched=True,
    remove_columns=dataset["train"].column_names
)

### Model Training

In [12]:
output_dir = "/cs/student/projects1/2022/shuhanho/best_phi_run"
os.makedirs(output_dir, exist_ok=True)

In [13]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  
)

In [15]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
)

In [16]:
training_args = TrainingArguments(
    per_device_train_batch_size=1,  
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=128,   
     
    max_steps=20, 
    resume_from_checkpoint = True, 
    
    warmup_steps=10,
    logging_steps=20,                 
    save_steps=20,                   
    save_total_limit=5,
    save_strategy="steps",
    eval_strategy="steps",
    
    load_best_model_at_end = True,  
    metric_for_best_model = "eval_loss",  
    greater_is_better = False,
        
    eval_steps=20,                   
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    output_dir = "/cs/student/projects1/2022/shuhanho/best_phi_run",
    report_to="wandb",
    run_name = "phi3.5-run",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    data_collator=data_collator,
    packing=False,
    dataset_text_field="input_ids",
    max_seq_length=2048,
    args=training_args,
)

In [17]:
import transformers
class WandbCallback(transformers.TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'learning_rate' in logs:
                wandb.log({"learning_rate": logs['learning_rate']})
            if 'loss' in logs:
                wandb.log({"train/loss": logs['loss']})
            if 'eval_loss' in logs:
                wandb.log({"eval/loss": logs['eval_loss']})

trainer.add_callback(WandbCallback())

In [18]:
accelerator = Accelerator()
trainer = accelerator.prepare(trainer)

In [19]:
print("\nStarting training...")
trainer_stats = unsloth_train(trainer)

print("\nTraining completed!")
print(f"Training time: {trainer_stats.metrics['train_runtime']:.2f} seconds")
print(f"Training loss: {trainer_stats.metrics['train_loss']:.4f}")
trainer.save_model("/cs/student/projects1/2022/shuhanho/best_phi_final_model")
wandb.finish()

### Inference

In [ ]:
from typing import List, Dict, Tuple, Any
import re

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Phi-3.5-mini-instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "auto"
)

tokenizer = get_chat_template(
    tokenizer,
    chat_template="phi-3",
    mapping={
        "role": "role",
        "content": "content",
        "user": "user",
        "assistant": "assistant"
    }
)

tokenizer.padding_side = 'left'

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

model.load_adapter(
    model_id="/cs/student/projects1/2022/shuhanho/best_phi_run/checkpoint-2160",
    adapter_name="default"
)

FastLanguageModel.for_inference(model)

In [14]:
from amr import AMR

class AMRValidator:
    @staticmethod
    def validate_structure(amr_str: str) -> Tuple[bool, str, Dict]:
        stats = {
            'depth': 0,
            'current_depth': 0,
            'var_count': 0,
            'nesting_levels': [],
            'max_depth_allowed': 15
        }
        
        try:
            amr_str = re.sub(r'\s+', ' ', amr_str.strip())
            
            stack = []
            for i, char in enumerate(amr_str):
                if char == '(':
                    stats['current_depth'] += 1
                    stats['depth'] = max(stats['depth'], stats['current_depth'])
                    stack.append(i)
                elif char == ')':
                    if not stack:
                        return False, "Unmatched closing parenthesis", stats
                    stack.pop()
                    stats['current_depth'] -= 1
            
            if stack:
                return False, "Unmatched opening parenthesis", stats
                
            if stats['current_depth'] != 0:
                return False, "Unbalanced parentheses", stats
                
            if stats['depth'] > stats['max_depth_allowed']:
                return False, f"Exceeded maximum depth of {stats['max_depth_allowed']}", stats
            
            if not (amr_str.startswith('(') and amr_str.endswith(')')):
                return False, "AMR must start with '(' and end with ')'", stats
                
            var_def_pattern = r'\([a-z][0-9]?\s*/\s*[a-zA-Z0-9-]+'
            if not re.search(var_def_pattern, amr_str):
                return False, "No valid variable definitions found", stats
                
            relation_pattern = r':[a-zA-Z][a-zA-Z0-9-]*\s+'
            if not re.search(relation_pattern, amr_str):
                return False, "No valid relation labels found", stats
            
            parsed_amr = AMR.parse_AMR_line(amr_str)
            
            if parsed_amr is None:
                return False, "Failed to parse AMR structure", stats
                
            if not parsed_amr.nodes:
                return False, "No nodes found in AMR", stats
                
            if not parsed_amr.node_values:
                return False, "No node values found in AMR", stats
                
            if not parsed_amr.root:
                return False, "No root node found in AMR", stats
                
            if len(parsed_amr.nodes) != len(parsed_amr.node_values):
                return False, "Mismatch between nodes and values", stats
            
            stats['var_count'] = len(parsed_amr.nodes)
            
            return True, amr_str, stats
            
        except Exception as e:
            return False, f"Error during validation: {str(e)}", stats

    @staticmethod
    def validate_semantics(amr: str) -> Dict[str, Any]:
        results = {
            'is_valid': False,
            'errors': [],
            'warnings': [],
            'details': {
                'has_root': False,
                'has_predicates': False,
                'valid_relations': True
            }
        }

        try:
            if not re.search(r'\(\s*[\w-]+\s*/', amr):
                results['errors'].append("No root concept found")
            else:
                results['details']['has_root'] = True

            if not re.search(r'-\d+\s*/', amr):
                results['warnings'].append("No numbered predicates found")
            else:
                results['details']['has_predicates'] = True

            relation_pattern = r':[A-Za-z][A-Za-z0-9-]*\s+'
            invalid_relations = re.findall(r':[^A-Za-z\s]|:[A-Za-z][^A-Za-z0-9-]*', amr)
            if invalid_relations:
                results['errors'].append(f"Invalid relations found: {invalid_relations}")
                results['details']['valid_relations'] = False

            results['is_valid'] = (not results['errors'] and
                                 results['details']['has_root'] and
                                 results['details']['valid_relations'])

        except Exception as e:
            results['errors'].append(f"Validation error: {str(e)}")
            results['is_valid'] = False

        return results

In [15]:
class SMATCHEvaluator:
    def __init__(self, smatch_script_path: str = "smatch.py"):
        self.smatch_script_path = smatch_script_path

    @staticmethod
    def save_amrs(filename: str, amrs: List[str], is_gold: bool = False) -> None:
        with open(filename, 'w', encoding='utf-8') as f:
            for i, amr in enumerate(amrs):
                f.write(f"# ::id EVAL.{i}\n")
                f.write(f"# ::snt DUMMY_SENT\n")
                f.write(f"# ::annotator {'gold' if is_gold else 'generated'}\n")
                f.write(amr.strip() + "\n\n")

    @staticmethod
    def parse_output(output: str) -> Tuple[float, float, float]:
        try:
            p_match = re.search(r"Precision:\s*([\d.]+)", output)
            r_match = re.search(r"Recall:\s*([\d.]+)", output)
            f_match = re.search(r"F-score:\s*([\d.]+)", output)

            if not all([p_match, r_match, f_match]):
                return 0.0, 0.0, 0.0

            return (float(p_match.group(1)),
                   float(r_match.group(1)),
                   float(f_match.group(1)))

        except Exception as e:
            print(f"Error parsing SMATCH output: {str(e)}")
            return 0.0, 0.0, 0.0

    def evaluate_pair(self, generated_amr: str, gold_amr: str, idx: int) -> Tuple[float, float, float]:
        gen_file = f"gen_{idx}.txt"
        gold_file = f"gold_{idx}.txt"

        try:
            self.save_amrs(gen_file, [generated_amr])
            self.save_amrs(gold_file, [gold_amr], is_gold=True)

            cmd = ["python", self.smatch_script_path,
                "-f", gen_file, gold_file,
                "--pr", "-r", "10"]

            output = subprocess.run(cmd,
                                capture_output=True,
                                text=True,
                                check=False)

            if output.returncode == 0:
                return self.parse_output(output.stdout)
            return 0.0, 0.0, 0.0

        except Exception:
            return 0.0, 0.0, 0.0

        finally:
            for f in [gen_file, gold_file]:
                try:
                    if os.path.exists(f):
                        os.remove(f)
                except:
                    pass

In [16]:
class AMRVisualizer:
    @staticmethod
    def create_depth_performance_plot(metrics_by_depth: Dict, output_dir: str = None) -> go.Figure:
        fig = make_subplots(
            rows=2, cols=1,
            subplot_titles=('SMATCH Scores by AMR Depth', 'Examples by Depth'),
            vertical_spacing=0.3
        )

        colors = {
            'F1': 'rgb(31, 119, 180)',
            'Precision': 'rgb(255, 127, 14)',
            'Recall': 'rgb(44, 160, 44)'
        }

        depths = metrics_by_depth['depths']
        metrics_clean = {
            'f1s': [],
            'precisions': [],
            'recalls': [],
            'valid_depths': []
        }

        for i, (f1, prec, rec) in enumerate(zip(metrics_by_depth['f1s'],
                                            metrics_by_depth['precisions'],
                                            metrics_by_depth['recalls'])):
            if f1 is not None and prec is not None and rec is not None:
                metrics_clean['f1s'].append(f1)
                metrics_clean['precisions'].append(prec)
                metrics_clean['recalls'].append(rec)
                metrics_clean['valid_depths'].append(depths[i])

        for metric, name in [('f1s', 'F1'),
                            ('precisions', 'Precision'),
                            ('recalls', 'Recall')]:
            fig.add_trace(
                go.Scatter(
                    x=metrics_clean['valid_depths'],
                    y=metrics_clean[metric],
                    name=name,
                    mode='lines+markers',
                    line=dict(color=colors[name], width=2),
                    marker=dict(size=8)
                ),
                row=1, col=1
            )

        fig.add_trace(
            go.Bar(
                x=metrics_by_depth['depths'],
                y=metrics_by_depth['counts'],
                name='Valid Generations',
                marker_color='rgb(158,202,225)'
            ),
            row=2, col=1
        )

        fig.add_trace(
            go.Bar(
                x=metrics_by_depth['depths'],
                y=metrics_by_depth['invalid_structural_counts'],
                name='Invalid Generations',
                marker_color='rgb(255,177,177)'
            ),
            row=2, col=1
        )

        fig.update_layout(
            showlegend=True,
            template='plotly_white',
            title_text="AMR Performance Analysis",
            legend=dict(
                yanchor="top",
                y=0.99,
                xanchor="right",
                x=0.99
            ),
            barmode='stack',  
            width=1000,
            height=800
        )

        fig.update_xaxes(title_text='AMR Depth', row=2, col=1)
        fig.update_yaxes(title_text='SMATCH Score', row=1, col=1)
        fig.update_yaxes(title_text='Number of Examples', row=2, col=1)

        fig.update_xaxes(tickmode='linear', tick0=1, dtick=1)
        
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
            html_path = os.path.join(output_dir, "depth_performance_plot.html")
            png_path = os.path.join(output_dir, "depth_performance_plot.png")
            pdf_path = os.path.join(output_dir, "depth_performance_plot.pdf")
        else:
            html_path = "depth_performance_plot.html"
            png_path = "depth_performance_plot.png"
            pdf_path = "depth_performance_plot.pdf"
            
        fig.write_html(html_path)
        print(f"HTML plot saved to {html_path}")
        
        fig.write_image(png_path)
        print(f"PNG plot saved to {png_path}")
        
        fig.write_image(pdf_path)
        print(f"PDF plot saved to {pdf_path}")
        
        return fig

    def print_summary_statistics(self, metrics_by_depth: Dict) -> None:
        total_examples = sum(metrics_by_depth['total_samples'])
        if total_examples == 0:
            print("No examples were evaluated!")
            return
            
        total_structurally_valid = sum(metrics_by_depth['counts'])
        total_structurally_invalid = sum(metrics_by_depth['invalid_structural_counts'])

        total_semantically_valid = sum(sm['valid_count'] for sm in metrics_by_depth['semantic_metrics'])
        total_has_root = sum(sm['has_root'] for sm in metrics_by_depth['semantic_metrics'])
        total_has_predicates = sum(sm['has_predicates'] for sm in metrics_by_depth['semantic_metrics'])
        total_valid_relations = sum(sm['valid_relations'] for sm in metrics_by_depth['semantic_metrics'])
        total_warnings = sum(sm['total_warnings'] for sm in metrics_by_depth['semantic_metrics'])
        total_errors = sum(sm['total_errors'] for sm in metrics_by_depth['semantic_metrics'])

        valid_f1_scores = [(f1, count) for f1, count in zip(metrics_by_depth['f1s'], 
                                                       metrics_by_depth['counts']) 
                      if f1 is not None]
        
        if valid_f1_scores:
            weighted_f1 = sum(f1 * count for f1, count in valid_f1_scores) / sum(count for _, count in valid_f1_scores)
        else:
            weighted_f1 = 0

        print("\nSummary Statistics:")
        print("=" * 50)
        print(f"Total examples evaluated: {total_examples}")

        print("\nStructural Validation:")
        print(f"Valid generations: {total_structurally_valid}")
        print(f"Invalid generations: {total_structurally_invalid}")
        print(f"Structural validity rate: {total_structurally_valid/total_examples:.1%}")

        print("\nSemantic Validation:")
        print(f"Semantically valid: {total_semantically_valid}")
        print(f"Semantic validity rate: {total_semantically_valid/total_examples:.1%}")
        print(f"Has root concept: {total_has_root/total_examples:.1%}")
        print(f"Has predicates: {total_has_predicates/total_examples:.1%}")
        print(f"Valid relations: {total_valid_relations/total_examples:.1%}")
        print(f"Average warnings per example: {total_warnings/total_examples:.2f}")
        print(f"Average errors per example: {total_errors/total_examples:.2f}")

        print(f"\nDepth range: {min(metrics_by_depth['depths'])} to {max(metrics_by_depth['depths'])}")
        print(f"Weighted average F1 score: {weighted_f1:.3f}")

        print("\nPerformance by depth:")
        for i, depth in enumerate(metrics_by_depth['depths']):
            total = metrics_by_depth['total_samples'][i]
            structurally_valid = metrics_by_depth['counts'][i]
            semantic_metrics = metrics_by_depth['semantic_metrics'][i]
            
            f1_score = metrics_by_depth['f1s'][i]
            precision = metrics_by_depth['precisions'][i]
            recall = metrics_by_depth['recalls'][i]

            print(f"\nDepth {depth} (total={total}):")
            print(f"  Structural validity rate: {structurally_valid/total:.1%}")
            print(f"  Semantic validity rate: {semantic_metrics['valid_count']/total:.1%}")
            print(f"  Has root concept: {semantic_metrics['has_root']/total:.1%}")
            print(f"  Has predicates: {semantic_metrics['has_predicates']/total:.1%}")
            print(f"  Valid relations: {semantic_metrics['valid_relations']/total:.1%}")
            print(f"  Average warnings: {semantic_metrics['total_warnings']/total:.2f}")
            print(f"  Average errors: {semantic_metrics['total_errors']/total:.2f}")
            if f1_score is not None:
                print(f"  F1: {f1_score:.3f}")
                print(f"  Precision: {precision:.3f}")
                print(f"  Recall: {recall:.3f}")
            else:
                print("  No valid SMATCH scores")

In [17]:
class AMREvaluator:
    def __init__(self, model, tokenizer, smatch_script_path: str = "smatch.py"):
        self.model = model
        self.tokenizer = tokenizer
        self.validator = AMRValidator()
        self.smatch_evaluator = SMATCHEvaluator(smatch_script_path)
        self.visualizer = AMRVisualizer()
        
    def extract_amr(self,generated_text: str) -> str:
        stack = []
        start_idx = -1

        amr_start_pattern = r'\([a-zA-Z][0-9]?\s*/\s*[a-zA-Z0-9-]+'
        match = re.search(amr_start_pattern, generated_text)
        
        if not match:
            return ""
            
        for i in range(match.start(), len(generated_text)):
            char = generated_text[i]
            
            if char == '(':
                if start_idx == -1:
                    start_idx = i
                stack.append(i)
            elif char == ')':
                if stack:
                    stack.pop()
                    if not stack:
                        amr_text = generated_text[start_idx:i+1].strip()
                        
                        if '/' in amr_text:  
                            amr_text = re.sub(r'\s+', ' ', amr_text)
                            return amr_text
                        
                        start_idx = -1
                else:
                    start_idx = -1
        
        return ""

    def generate_amr(self, text: str) -> str:
        messages = [
            {"role": "system", "content": "You are an AMR parser. Convert English sentences into Abstract Meaning Representation (AMR) graphs. Use proper AMR notation and formatting."},
            {"role": "user", "content": text}
        ]
        
        input_ids = self.tokenizer.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")
        
        attention_mask = (input_ids != self.tokenizer.pad_token_id).to("cuda")
        
        outputs = self.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=2048,
            temperature=0.7,      
            top_p=1.0,           
            repetition_penalty=1.0,
            do_sample=True,      
            use_cache=True,
            num_return_sequences=1,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        
        full_output = self.tokenizer.decode(outputs[0], skip_special_tokens=False)
        
        generated_text = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"Cleaned output: {generated_text}\n")
        
        try:
            amr_text = self.extract_amr(generated_text)
            print(f"Extracted AMR: {amr_text}")
            if not amr_text:
                return ""
                
            
            return amr_text
                
        except Exception as e:
            print(f"Error extracting AMR: {str(e)}")
            return ""

    def _collect_examples_by_depth(self,
                           dataset,
                           max_depth: int = 10,
                           samples_per_depth: int = 1) -> Dict:
        examples_by_depth = {d: [] for d in range(1, max_depth + 1)}
        
        dataset = list(dataset)
        random.shuffle(dataset)

        for example in dataset:
            conversations = example["conversations"]
            if len(conversations) < 2:
                continue
                
            amr = conversations[1]["content"].strip()

            is_valid, _, stats = self.validator.validate_structure(amr)
            if not is_valid:
                continue

            depth = int(round(stats['depth']))
            
            if depth <= max_depth and len(examples_by_depth[depth]) < samples_per_depth:
                examples_by_depth[depth].append(example)
                
                if all(len(examples_by_depth[d]) >= samples_per_depth 
                    for d in range(1, max_depth + 1)):
                    break

        return examples_by_depth
    
    def _evaluate_examples(self, examples_by_depth: Dict) -> Dict:
        metrics = {
            'depths': [],
            'f1s': [],
            'precisions': [],
            'recalls': [],
            'counts': [],          
            'total_samples': [],   
            'invalid_structural_counts': [],
            'semantic_metrics': [],
        }

        total_examples = sum(len(examples) for examples in examples_by_depth.values())
        pbar = tqdm(total=total_examples, desc="Evaluating AMRs")

        sorted_depths = sorted(examples_by_depth.keys())

        for depth in sorted_depths:
            examples = examples_by_depth[depth]
            if not examples:
                continue

            depth_scores = {
                'precision': [],
                'recall': [],
                'f1': [],
                'invalid_structural': 0,
                'semantic_results': {
                    'valid_count': 0,
                    'has_root': 0,
                    'has_predicates': 0,
                    'valid_relations': 0,
                    'total_warnings': 0,
                    'total_errors': 0
                }
            }

            for idx, example in enumerate(examples):
                conversations = example["conversations"]
                if len(conversations) < 2:
                    continue
                    
                input_text = conversations[0]["content"].replace(
                    "Generate an Abstract Meaning Representation (AMR) graph for the following sentence: ",
                    ""
                ).strip()
                gold_amr = conversations[1]["content"].strip()

                generated_amr = self.generate_amr(input_text)
                is_structurally_valid, generated_amr, _ = self.validator.validate_structure(generated_amr)

                semantic_results = self.validator.validate_semantics(generated_amr)

                if semantic_results['is_valid']:
                    depth_scores['semantic_results']['valid_count'] += 1
                if semantic_results['details']['has_root']:
                    depth_scores['semantic_results']['has_root'] += 1
                if semantic_results['details']['has_predicates']:
                    depth_scores['semantic_results']['has_predicates'] += 1
                if semantic_results['details']['valid_relations']:
                    depth_scores['semantic_results']['valid_relations'] += 1
                depth_scores['semantic_results']['total_warnings'] += len(semantic_results['warnings'])
                depth_scores['semantic_results']['total_errors'] += len(semantic_results['errors'])

                if not is_structurally_valid:
                    depth_scores['invalid_structural'] += 1
                    print(f"Invalid structure - skipping SMATCH computation")
                    pbar.update(1)
                    continue

                max_retries = 3
                smatch_success = False
                
                for attempt in range(max_retries):
                    try:
                        precision, recall, f1 = self.smatch_evaluator.evaluate_pair(
                            generated_amr,
                            gold_amr,
                            f"d{depth}_{idx}"
                        )
                        if all(0 <= score <= 1 for score in [precision, recall, f1]):
                            depth_scores['precision'].append(precision)
                            depth_scores['recall'].append(recall)
                            depth_scores['f1'].append(f1)
                            smatch_success = True
                            print(f"SMATCH scores - P: {precision:.3f}, R: {recall:.3f}, F1: {f1:.3f}")
                            break
                        else:
                            print(f"Invalid SMATCH scores computed: P={precision}, R={recall}, F1={f1}")
                    except Exception as e:
                        if attempt == max_retries - 1:
                            print(f"Failed to compute SMATCH scores after {max_retries} attempts: {str(e)}")
                            print(f"Generated AMR: {generated_amr}")
                            print(f"Gold AMR: {gold_amr}")

                pbar.update(1)

            if examples:
                metrics['depths'].append(depth)
                valid_count = len(examples) - depth_scores['invalid_structural']
                
                if depth_scores['precision']:  
                    avg_precision = np.mean(depth_scores['precision'])
                    avg_recall = np.mean(depth_scores['recall'])
                    avg_f1 = np.mean(depth_scores['f1'])
                    
                    metrics['precisions'].append(avg_precision)
                    metrics['recalls'].append(avg_recall)
                    metrics['f1s'].append(avg_f1)
                else:
                    metrics['precisions'].append(0.0)
                    metrics['recalls'].append(0.0)
                    metrics['f1s'].append(0.0)
                    
                metrics['counts'].append(valid_count)
                metrics['total_samples'].append(len(examples))
                metrics['invalid_structural_counts'].append(depth_scores['invalid_structural'])
                metrics['semantic_metrics'].append(depth_scores['semantic_results'])

        pbar.close()
        return metrics


    def analyze_depth_distribution(self, dataset) -> Dict[int, int]:

        depth_distribution = {}
        print("Analyzing depth distribution...")

        for example in tqdm(dataset, desc="Analyzing depths"):
            conversations = example["conversations"]
            if len(conversations) < 2:
                continue
                
            amr = conversations[1]["content"].strip()

            is_valid, _, stats = self.validator.validate_structure(amr)
            if not is_valid:
                continue

            depth = int(round(stats['depth']))
            depth_distribution[depth] = depth_distribution.get(depth, 0) + 1

        return depth_distribution

    def evaluate_balanced_depths(self, 
                               dataset, 
                               samples_per_depth: int = 1,
                               max_depth: int = 10,
                               min_depth: int = 1,
                               visualize: bool = True) -> Dict:
        depth_dist = self.analyze_depth_distribution(dataset)
        total_examples = sum(depth_dist.values())
        for depth in sorted(depth_dist.keys()):
            if depth >= min_depth: 
                count = depth_dist[depth]
                percentage = (count / total_examples) * 100
                print(f"Depth {depth}: {count} examples ({percentage:.1f}%)")
                
        examples = {}
        total_examples = 0
        
        print("\nCollecting examples by depth:")
        print("=" * 50)
        for depth in range(min_depth, max_depth + 1):
            if depth not in depth_dist:
                continue
                
            examples_list = self._collect_examples_by_depth(
                dataset, 
                max_depth=depth,
                samples_per_depth=samples_per_depth
            )[depth]
            
            if examples_list:
                examples[depth] = examples_list
                total_examples += len(examples_list)
                print(f"Depth {depth}: {len(examples_list)} examples")
        
        print("\nEvaluating examples...")
        metrics = self._evaluate_examples(examples)
        
        if visualize:
            fig = self.visualizer.create_depth_performance_plot(metrics, "/cs/student/projects1/2022/shuhanho/silver_phi_plots")
            fig.show()
            self.visualizer.print_summary_statistics(metrics)
        
        return metrics

    def evaluate_single(self,
                       input_text: str,
                       gold_amr: str = None) -> Dict[str, Any]:

        results = {
            'generated_amr': None,
            'validation': None,
            'smatch_scores': None,
            'structure_stats': None
        }

        generated_amr = self.generate_amr(input_text)
        results['generated_amr'] = generated_amr

        is_valid, validated_amr, stats = self.validator.validate_structure(generated_amr)
        results['structure_stats'] = stats

        semantic_validation = self.validator.validate_semantics(generated_amr)
        results['validation'] = {
            'structural': is_valid,
            'semantic': semantic_validation
        }

        if gold_amr and is_valid:
            precision, recall, f1 = self.smatch_evaluator.evaluate_pair(
                validated_amr,
                gold_amr,
                'single'
            )
            results['smatch_scores'] = {
                'precision': precision,
                'recall': recall,
                'f1': f1
            }

        return results

### Full Dataset Inference

In [ ]:
evaluator = AMREvaluator(model, tokenizer)

examples_by_depth = {}
for example in dataset["test"]:
    text = example["text"]
    amr = text.split("### Response:\n")[1].split("<eos>")[0].strip()
    
    is_valid, _, stats = AMRValidator.validate_structure(amr)
    depth = stats['depth']
    
    if depth not in examples_by_depth:
        examples_by_depth[depth] = []
    examples_by_depth[depth].append(example)

metrics = evaluator._evaluate_examples(examples_by_depth)

print("\nFull Dataset Results:")
print(f"Total examples evaluated: {sum(metrics['total_samples'])}")
print(f"Valid generations: {sum(metrics['counts'])}")
print(f"Invalid structural generations: {sum(metrics['invalid_structural_counts'])}")
print(f"\nOverall metrics:")
print(f"Average F1: {np.mean(metrics['f1s']):.3f}")
print(f"Average Precision: {np.mean(metrics['precisions']):.3f}")
print(f"Average Recall: {np.mean(metrics['recalls']):.3f}")

print("\nResults by depth:")
for i, depth in enumerate(metrics['depths']):
    print(f"\nDepth {depth}:")
    print(f"  Samples: {metrics['total_samples'][i]}")
    print(f"  Valid: {metrics['counts'][i]}")
    print(f"  Invalid: {metrics['invalid_structural_counts'][i]}")
    print(f"  F1: {metrics['f1s'][i]:.3f}")
    print(f"  Precision: {metrics['precisions'][i]:.3f}")
    print(f"  Recall: {metrics['recalls'][i]:.3f}")

### Individual Inference

In [29]:
evaluator = AMREvaluator(model, tokenizer)

metrics = evaluator.evaluate_balanced_depths(
    dataset["test"],
    samples_per_depth=30,
    max_depth=10,
    visualize=True
)

### Silver Data Inference

In [ ]:
from process_silver_amr import load_silver_dataset

silver_dataset = load_silver_dataset()
formatted_dataset = silver_dataset.map(formatting_prompts_func, batched=True)

evaluator = AMREvaluator(model, tokenizer)
metrics = evaluator.evaluate_balanced_depths(
    formatted_dataset,
    samples_per_depth=16,
    max_depth=3,
    min_depth=2,
    visualize=True
)

### Comparative Inference

In [ ]:
from process_test_amr import process_test_files

test_datasets = process_test_files()
all_metrics = {}

for dataset_name, dataset in test_datasets.items():
    print(f"\nEvaluating {dataset_name} dataset...")
    formatted_dataset = dataset.map(formatting_prompts_func, batched=True)
    
    evaluator = AMREvaluator(model, tokenizer)
    
    metrics = evaluator.evaluate_balanced_depths(
        formatted_dataset,
        samples_per_depth=10,  
        max_depth=10,
        visualize=False 
    )
    
    all_metrics[dataset_name] = metrics


In [20]:
def create_comparative_depth_plots(all_metrics):
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('SMATCH Scores by AMR Depth Across Datasets',
                       'Sample Distribution by Depth Across Datasets'),
        vertical_spacing=0.3
    )
    
    dataset_colors = {
        name: color for name, color in zip(
            all_metrics.keys(),
            ['rgb(31, 119, 180)', 'rgb(255, 127, 14)', 
             'rgb(44, 160, 44)', 'rgb(214, 39, 40)',
             'rgb(148, 103, 189)', 'rgb(140, 86, 75)']
        )
    }
    
    metric_styles = {
        'F1': 'solid',
        'Precision': 'dash',
        'Recall': 'dot'
    }
    
    for dataset_name, metrics in all_metrics.items():
        base_color = dataset_colors[dataset_name]
        
        fig.add_trace(
            go.Scatter(
                x=metrics['depths'],
                y=metrics['f1s'],
                name=f'{dataset_name} F1',
                mode='lines+markers',
                line=dict(color=base_color, width=2, dash=metric_styles['F1']),
                marker=dict(size=8)
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=metrics['depths'],
                y=metrics['precisions'],
                name=f'{dataset_name} Precision',
                mode='lines+markers',
                line=dict(color=base_color, width=2, dash=metric_styles['Precision']),
                marker=dict(size=8)
            ),
            row=1, col=1
        )
        
        fig.add_trace(
            go.Scatter(
                x=metrics['depths'],
                y=metrics['recalls'],
                name=f'{dataset_name} Recall',
                mode='lines+markers',
                line=dict(color=base_color, width=2, dash=metric_styles['Recall']),
                marker=dict(size=8)
            ),
            row=1, col=1
        )
    
    all_depths = sorted(set(
        depth for metrics in all_metrics.values() 
        for depth in metrics['depths']
    ))
    
    bar_width = 0.8 / len(all_metrics)  
    
    for depth in all_depths:
        for idx, (dataset_name, metrics) in enumerate(all_metrics.items()):
            if depth in metrics['depths']:
                depth_idx = metrics['depths'].index(depth)
                x_pos = depth + (idx - len(all_metrics)/2 + 0.5) * bar_width
                
                fig.add_trace(
                    go.Bar(
                        x=[x_pos],
                        y=[metrics['counts'][depth_idx]],
                        name=f'{dataset_name} Valid',
                        marker_color=dataset_colors[dataset_name],
                        opacity=0.7,
                        width=bar_width,
                        showlegend=depth == all_depths[0]  
                    ),
                    row=2, col=1
                )
                
                fig.add_trace(
                    go.Bar(
                        x=[x_pos],
                        y=[metrics['invalid_structural_counts'][depth_idx]],
                        name=f'{dataset_name} Invalid',
                        marker_color=dataset_colors[dataset_name],
                        opacity=0.3,
                        width=bar_width,
                        showlegend=depth == all_depths[0]  
                    ),
                    row=2, col=1
                )

    fig.update_layout(
        height=1000,
        showlegend=True,
        template='plotly_white',
        title_text="AMR Performance Analysis Across Datasets",
        barmode='stack',
        legend=dict(
            yanchor="top",
            y=0.99,
            xanchor="right",
            x=0.99
        )
    )

    fig.update_xaxes(title_text='AMR Depth', row=2, col=1)
    fig.update_yaxes(title_text='SMATCH Score', row=1, col=1)
    fig.update_yaxes(title_text='Number of Examples', row=2, col=1)
    
    fig.update_xaxes(
        tickmode='array',
        tickvals=all_depths,
        ticktext=[str(d) for d in all_depths],
        row=2, col=1
    )
    
    
    output_dir = "/cs/student/projects1/2022/shuhanho/comparative_phi_plots"
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        html_path = os.path.join(output_dir, "depth_performance_plot.html")
        png_path = os.path.join(output_dir, "depth_performance_plot.png")
        pdf_path = os.path.join(output_dir, "depth_performance_plot.pdf")
    else:
        html_path = "depth_performance_plot.html"
        png_path = "depth_performance_plot.png"
        pdf_path = "depth_performance_plot.pdf"
        
    fig.write_html(html_path)
    print(f"HTML plot saved to {html_path}")
    
    fig.write_image(png_path)
    print(f"PNG plot saved to {png_path}")
    
    fig.write_image(pdf_path)
    print(f"PDF plot saved to {pdf_path}")

    return fig

In [ ]:
comparative_fig = create_comparative_depth_plots(all_metrics)
comparative_fig.show()

print("\nComparative Statistics Across Datasets:")
print("=" * 50)
for dataset_name, metrics in all_metrics.items():
    print(f"\n{dataset_name.upper()} Dataset:")
    total_examples = sum(metrics['total_samples'])
    total_valid = sum(metrics['counts'])
    weighted_f1 = sum(f1 * count for f1, count in 
                     zip(metrics['f1s'], metrics['counts'])) / total_valid
    
    print(f"Total examples: {total_examples}")
    print(f"Valid generations: {total_valid}")
    print(f"Validity rate: {total_valid/total_examples:.1%}")
    print(f"Weighted average F1: {weighted_f1:.3f}")